In [112]:
import numpy as np
import pandas as pd
from functools import lru_cache
from itertools import product

# =======================
# Parâmetros do problema
# =======================
#wells = ("A","B","C","D","E","F","G","H")
wells = ("A","B","C","D")  # em vez de 8
k = 4  # estados #0..#3
R = np.array([-10, -5, 5, 10], dtype=float)  # Tabela 2
IC = {w: 0.0 for w in wells}                  # custo de informação
delta = 1.0

# Marginais por poço (Tabela 3)
P_marg = {
    "A": [0.02, 0.20, 0.36, 0.42],
    "B": [0.19, 0.24, 0.38, 0.19],
    "C": [0.05, 0.22, 0.50, 0.23],
    "D": [0.04, 0.44, 0.30, 0.23],
    "E": [0.26, 0.38, 0.23, 0.13],
    "F": [0.12, 0.44, 0.34, 0.11],
    "G": [0.09, 0.42, 0.34, 0.15],
    "H": [0.38, 0.31, 0.16, 0.16],
}

# Marginal "da campanha" (linha Campaign da Tabela 3)
P_camp = np.array([0.14, 0.33, 0.33, 0.20], dtype=float)

# ============================================================
# 1) Constrói um JPD com dependência via fator latente global
#    Xi | Q=q  ~  (1-ρ)*P_i  +  ρ*one_hot(q)
#    J(x1..xN) = sum_q P(Q=q) * prod_i P(Xi=xi | q)
# ============================================================
def build_correlated_J(wells, P_marg, P_camp, rho=0.4, noise=0.05):
    pm = [np.array(P_marg[w], dtype=float) for w in wells]
    pm = [p/p.sum() for p in pm]  # normaliza
    J = np.zeros((k,)*len(wells), dtype=float)
    for idx, x in enumerate(product(range(k), repeat=len(wells))):
        prob = 0.0
        for q in range(k):
            pq = P_camp[q]
            term = 1.0
            for i, xi in enumerate(x):
                pxi_q = (1.0 - rho) * pm[i][xi] + rho * (1.0 if xi == q else 0.0)
                term *= pxi_q
            prob += pq * term
        J[x] = prob
    # adiciona ruído
    J *= 1 + np.random.normal(0, noise, J.shape)
    J[J < 0] = 0
    J /= J.sum()
    return J

J = build_correlated_J(wells, P_marg, P_camp, rho=0.40)  # ajustável

# ================================
# 2) Utilidades (Bayes + marginais)
# ================================
def marginal(J, axis):
    axes = tuple(i for i in range(J.ndim) if i != axis)
    return J.sum(axis=axes)

def posterior(J, axis, obs):
    slicer = [slice(None)]*J.ndim
    slicer[axis] = obs
    num = J[tuple(slicer)]
    p = num.sum()
    if p <= 0:
        return None, 0.0
    post = num / p
    return post, p

# ========================================================
# 3) Bellman (SDP) — retorna (valor ótimo, 1ª ação ótima)
# ========================================================
@lru_cache(maxsize=None)
def best_action(state_wells, J_bytes):
    rem = list(state_wells)
    if not rem:
        return 0.0, None
    J_state = np.frombuffer(J_bytes, dtype=np.float64)
    m = int(round(np.log(J_state.size)/np.log(k)))
    J_state = J_state.reshape((k,)*m)

    best_val, best_first = -1e18, None
    for i, w in enumerate(rem):
        exp_val = 0.0
        for x in range(k):
            J_post, px = posterior(J_state, i, x)
            if px == 0: 
                continue
            v_next, _ = best_action(tuple(rem[:i]+rem[i+1:]), 
                                    J_post.astype(np.float64).tobytes())
            exp_val += px * ((R[x] - IC[w]) + delta * v_next)
        if exp_val > best_val:
            best_val, best_first = exp_val, w

    if best_val < 0:  # opção de parar
        return 0.0, None
    return best_val, best_first

# ==========================================================
# 4) Probabilidade de perfuração por posição (Tabela 5)
#    Percorre a política ótima e acumula massa de prob.
# ==========================================================
def policy_drill_probs(wells, J):
    probs = {w: np.zeros(len(wells)) for w in wells}
    def recurse(state, Jcur, pos, mass):
        val, first = best_action(state, Jcur.astype(np.float64).tobytes())
        if first is None or pos >= len(wells) or mass <= 0:
            return
        i = list(state).index(first)
        probs[first][pos] += mass
        # ramifica por todos os resultados possíveis do poço escolhido
        for x in range(k):
            J_post, px = posterior(Jcur, i, x)
            if px == 0: 
                continue
            recurse(tuple(w for w in state if w != first), J_post, pos+1, mass*px)

    recurse(tuple(wells), J, 0, 1.0)
    return probs

# =========================
# 5) Executa e imprime tudo
# =========================
V_opt, _ = best_action(wells, J.astype(np.float64).tobytes())

# 🔹 Função para reconstruir a sequência ótima completa
def optimal_sequence(wells, J):
    seq = []
    state = tuple(wells)
    Jcur = J.copy()
    while True:
        v, first = best_action(state, Jcur.astype(np.float64).tobytes())
        if first is None:
            break
        seq.append(first)

        # Atualiza a distribuição posterior (média ponderada dos possíveis resultados)
        i = list(state).index(first)
        Jnext = np.zeros_like(Jcur)
        for x in range(k):
            J_post, px = posterior(Jcur, i, x)
            if px == 0:
                continue
            Jnext += px * J_post
        Jcur = Jnext / Jnext.sum()
        state = tuple(w for w in state if w != first)

        # Parar se o valor esperado cair para zero (decisão de parar)
        v_next, _ = best_action(state, Jcur.astype(np.float64).tobytes())
        if v_next <= 1e-6:
            break

    return seq

# 🔹 Sequência ótima
seq_opt = optimal_sequence(wells, J)

print(f"Optimal Expected Value (EV*): {V_opt:.2f}")
print("🔹 Sequência ótima de perfuração:")
print(" → ".join(seq_opt))

# 🔹 Tabela de probabilidades (Tabela 5)
probs = policy_drill_probs(wells, J)
df = pd.DataFrame({w: 100 * probs[w] for w in wells},
                  index=range(1, len(wells)+1))
df.loc["Total"] = df.sum()
print("\nProbabilidade de perfuração por posição (Tabela 5 simulada):")
print(df.round(2))



Optimal Expected Value (EV*): 7.94
🔹 Sequência ótima de perfuração:
A → B → C

Probabilidade de perfuração por posição (Tabela 5 simulada):
           A      B      C      D
1      100.0   0.00   0.00   0.00
2        0.0   0.00  93.24   0.00
3        0.0   9.72   0.00  67.49
4        0.0  50.44   0.00   9.72
Total  100.0  60.16  93.24  77.21


In [113]:
def print_policy_level1_and_2(wells, J):
    state = tuple(wells)
    val, first = best_action(state, J.astype(np.float64).tobytes())
    if first is None:
        print("Stop.")
        return
    print(f"1ª decisão ótima: perfurar {first} (EV* = {val:.2f})")

    i = list(state).index(first)
    for x in range(k):
        J_post, px = posterior(J, i, x)
        if px == 0: 
            continue
        nxt_state = tuple(w for w in state if w != first)
        v_next, second = best_action(nxt_state, J_post.astype(np.float64).tobytes())
        act2 = second if second is not None else "Stop"
        print(f"  Se {first} = #{x} (P={px:.3f}) → próxima ação: {act2} (V={v_next:.2f})")

print_policy_level1_and_2(wells, J)


1ª decisão ótima: perfurar A (EV* = 7.94)
  Se A = #0 (P=0.068) → próxima ação: Stop (V=0.00)
  Se A = #1 (P=0.251) → próxima ação: C (V=1.57)
  Se A = #2 (P=0.351) → próxima ação: C (V=6.18)
  Se A = #3 (P=0.331) → próxima ação: C (V=6.77)


In [98]:
import numpy.random as npr

def sample_sequence(wells, J, seed=None, maxlen=None):
    rng = npr.default_rng(seed)
    state = tuple(wells)
    Jcur = J.copy()
    seq = []
    total_ev = 0.0
    steps = 0
    while True:
        v_now, w = best_action(state, Jcur.astype(np.float64).tobytes())
        if w is None:
            break
        seq.append(w)
        i = list(state).index(w)

        # marginais P(x|ω) para o poço escolhido
        p_vec = marginal(Jcur, i)
        p_vec = p_vec / p_vec.sum()

        # amostra um resultado realizável
        x = rng.choice(np.arange(k), p=p_vec)

        # EV imediato daquele resultado + valor futuro ótimo
        J_post, px = posterior(Jcur, i, x)
        v_next, _ = best_action(tuple(s for s in state if s != w),
                                J_post.astype(np.float64).tobytes())
        total_ev += (R[x] - IC[w]) + v_next

        # atualiza estado e JPD para o ramo observado
        Jcur = J_post
        state = tuple(s for s in state if s != w)

        steps += 1
        if maxlen and steps >= maxlen:
            break
    return seq, total_ev

# Exemplo: gerar 5 sequências possíveis
for t in range(5):
    seq_sim, ev_sim = sample_sequence(wells, J, seed=1234+t)
    print(f"Sequência simulada {t+1}: {' → '.join(seq_sim)} | EV realizado ≈ {ev_sim:.2f}")


Sequência simulada 1: A → C → D → B | EV realizado ≈ 33.73
Sequência simulada 2: A → C | EV realizado ≈ -8.46
Sequência simulada 3: A → C → D → B | EV realizado ≈ 17.09
Sequência simulada 4: A → C → D → B | EV realizado ≈ 6.25
Sequência simulada 5: A → C | EV realizado ≈ 6.80


In [99]:
def greedy_best_branch_sequence(wells, J):
    seq = []
    state = tuple(wells)
    Jcur = J.copy()
    while True:
        v, w = best_action(state, Jcur.astype(np.float64).tobytes())
        if w is None:
            break
        seq.append(w)
        i = list(state).index(w)

        # escolhe o resultado x* que dá a MAIOR continuação
        best_gain, best_Jpost = -1e18, None
        for x in range(k):
            J_post, px = posterior(Jcur, i, x)
            if px == 0: 
                continue
            v_next, _ = best_action(tuple(s for s in state if s != w),
                                    J_post.astype(np.float64).tobytes())
            gain = (R[x] - IC[w]) + v_next
            if gain > best_gain:
                best_gain, best_Jpost = gain, J_post

        if best_Jpost is None:
            break
        Jcur = best_Jpost
        state = tuple(s for s in state if s != w)
    return seq

print("Sequência (melhor ramo em cada passo):", " → ".join(greedy_best_branch_sequence(wells, J)))


Sequência (melhor ramo em cada passo): A → C → D → B
